## Setup Spark Session

In [4]:

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Lab01_Enhanced") \
    .getOrCreate()


In [ ]:
from pyspark.sql import SparkSession

spark =SparkSession.builder \
        .appName("odai") \
        .getOrCreate()

## Create DataFrame from Dictionary

In [5]:

data = [
    {"id": 1, "name": "Ali", "age": 25, "salary": 4000},
    {"id": 2, "name": "Sara", "age": 30, "salary": 7000},
    {"id": 3, "name": "Omar", "age": 22, "salary": None},
    {"id": 4, "name": "Mona", "age": None, "salary": 5000}
]

df = spark.createDataFrame(data)

df.show()
df.printSchema()

+----+---+----+------+
| age| id|name|salary|
+----+---+----+------+
|  25|  1| Ali|  4000|
|  30|  2|Sara|  7000|
|  22|  3|Omar|  null|
|null|  4|Mona|  5000|
+----+---+----+------+

root
 |-- age: long (nullable = true)
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)




## 🔍 Debugging Tip
Always inspect schema using:
- printSchema()
- show()

Why?
Because wrong data types will silently break your logic later.


## Filtering Data

    show salaries greater than 4000 and age > 23

In [9]:
df_salaries = df.filter((df["salary"] > 4000) & (df["age"] > 23))
df_salaries.show()

+---+---+----+------+
|age| id|name|salary|
+---+---+----+------+
| 30|  2|Sara|  7000|
+---+---+----+------+



## Renaming Columns

rename name column to be full_name

In [14]:
df_new = df.withColumnRenamed("name","full_name")
df_new.show()

+----+---+---------+------+
| age| id|full_name|salary|
+----+---+---------+------+
|  25|  1|      Ali|  4000|
|  30|  2|     Sara|  7000|
|  22|  3|     Omar|  null|
|null|  4|     Mona|  5000|
+----+---+---------+------+



## Reading CSV with proper schema

In [39]:
path = "file:///home/itversity/itversity-material/spark-labs/lab/employees.csv"

df= spark.read.csv(path, header=True, inferSchema=True)
df.show()


+----------+---------+---------+--------------------+----+------+----------------+----------+-------------------+------------+
|EmployeeID|FirstName| LastName|               Email| Age|Salary|      Department|  Position|           HireDate|        City|
+----------+---------+---------+--------------------+----+------+----------------+----------+-------------------+------------+
|         1|     John|    Smith|john.smith@compan...|  34| 75000|              IT|    Senior|2023-05-15 00:00:00|    New York|
|         2|     Emma|  Johnson|emma.johnson@comp...|null| 58000|              HR| Associate|2024-01-20 00:00:00| Los Angeles|
|         3|  Michael| Williams|michael.williams@...|  42| 92000|         Finance|   Manager|2022-11-03 00:00:00|     Chicago|
|         4|    Sarah|    Brown|sarah.brown@compa...|  28|  null|       Marketing|    Junior|2024-06-10 00:00:00|     Houston|
|         5|    David|    Jones|david.jones@compa...|null|112000|           Sales|  Director|2021-08-22 00:00:0


## 🧪 Data Quality Check

Find:
- Rows with null salary
- Rows with null age


In [43]:
df_null_sallary = df.filter(df["Salary"].isNull())
df_null_ages = df.filter(df["Age"].isNull())
df_null_sallary.show()
df_null_ages.show()

+----------+---------+--------+--------------------+----+------+----------+----------+-------------------+------------+
|EmployeeID|FirstName|LastName|               Email| Age|Salary|Department|  Position|           HireDate|        City|
+----------+---------+--------+--------------------+----+------+----------+----------+-------------------+------------+
|         4|    Sarah|   Brown|sarah.brown@compa...|  28|  null| Marketing|    Junior|2024-06-10 00:00:00|     Houston|
|         8|    Maria|   Davis|maria.davis@compa...|null|  null|       R&D|    Senior|2023-03-18 00:00:00|   San Diego|
|        14| Patricia|  Thomas|patricia.thomas@c...|null|  null|        HR| Associate|2023-11-25 00:00:00|     Houston|
|        20|  Jessica|  Harris|jessica.harris@co...|null|  null|        IT| Associate|2023-10-15 00:00:00|      Austin|
|        25|   Andrew|    Hall|andrew.hall@compa...|  30|  null|Operations|Specialist|2023-12-03 00:00:00|     Phoenix|
|        30|   Ashley|   Scott|ashley.sc

Handle null values:
- Replace null salary with 0
- Drop rows where age is null

In [46]:
df_replaced = df.na.fill({"Salary":0})
df_droped = df_replaced.na.drop(subset=["Age"]) 

In [51]:
df_droped.show()

+----------+---------+---------+--------------------+---+------+----------------+----------+-------------------+------------+
|EmployeeID|FirstName| LastName|               Email|Age|Salary|      Department|  Position|           HireDate|        City|
+----------+---------+---------+--------------------+---+------+----------------+----------+-------------------+------------+
|         1|     John|    Smith|john.smith@compan...| 34| 75000|              IT|    Senior|2023-05-15 00:00:00|    New York|
|         3|  Michael| Williams|michael.williams@...| 42| 92000|         Finance|   Manager|2022-11-03 00:00:00|     Chicago|
|         4|    Sarah|    Brown|sarah.brown@compa...| 28|     0|       Marketing|    Junior|2024-06-10 00:00:00|     Houston|
|         6|     Lisa|   Garcia|lisa.garcia@compa...| 31| 67000|      Operations|Specialist|2023-09-14 00:00:00|Philadelphia|
|         7|    James|   Miller|james.miller@comp...| 45| 89000|              IT|      Lead|2020-12-01 00:00:00| San A